# Jellyfish Detection

Single self-contained notebook: **setup → dataset config → train → predict → count to CSV**.

**Model:** 5 classes, trained with `yolo11n.pt`.

> **Training time:** Jellyfish uses the **nano** model (`yolo11n`).


## 1. Setup
Install and import the one library we need.

In [13]:
# Run once per environment. Safe to re-run.
%pip install -q ultralytics


Note: you may need to restart the kernel to use updated packages.


In [14]:

from ultralytics import YOLO

## 2. Dataset config
Writes `jellyfish.yaml` with **relative** paths, so this works on any machine
(no hardcoded `C:\\Users\\...` paths). Expects `images/train`, `images/val`
and matching `labels/` folders next to this notebook.

In [15]:
dataset_yaml = """
path: .                 # dataset root = this folder
train: images/train     # training images
val: images/val         # validation images

names:
  0: arctic_comb
  1: common_northern_comb
  2: lions_mane
  3: melon_comb
  4: moon_jelly
"""

with open("jellyfish.yaml", "w") as f:
    f.write(dataset_yaml)

print("Wrote jellyfish.yaml")

Wrote jellyfish.yaml


## 3. Train
Best weights are saved to `runs/detect/jellyfish_detection_model/weights/best.pt`.

In [16]:
model = YOLO("yolo11n.pt")   # start from the pretrained model

model.train(
    data="jellyfish.yaml",  # dataset config written above
    epochs=1000,             # at least 250 passes over the data
    imgsz=640,              # input image size
    batch=4,                # images per step (lower if memory is tight)
    device=0,           # set to 0 if you have a GPU (much faster)
    name="jellyfish_detection_model",

    # Stop early only if the val score stalls for 60 epochs.
    patience=60,
    close_mosaic=10,        # last 10 epochs train on normal images
)

print("Done. Best weights at: runs/detect/jellyfish_detection_model/weights/best.pt")

New https://pypi.org/project/ultralytics/8.4.126 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.124 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (NVIDIA RTX A6000, 48542MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=jellyfish.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1000, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, mode

## 4. Predict (visual check)
Run the trained model on the validation images and save annotated copies
so you can eyeball how it's doing. Output goes to a new `runs/detect/predict*` folder.

In [17]:
best_model = YOLO("runs/detect/jellyfish_detection_model/weights/best.pt")

best_model.predict(
    source="images/val",   # folder of images to test
    conf=0.5,              # drop weak detections
    iou=0.45,              # overlap threshold for merging boxes
    save=True,             # save annotated images to disk
)

print("Annotated predictions saved under runs/detect/")


image 1/66 /home/priyanshu/project/jellyfish/images/val/029416879.jpg: 384x640 1 moon_jelly, 7.0ms
image 2/66 /home/priyanshu/project/jellyfish/images/val/029491747.jpg: 384x640 (no detections), 7.6ms
image 3/66 /home/priyanshu/project/jellyfish/images/val/029889325.jpg: 384x640 (no detections), 8.1ms
image 4/66 /home/priyanshu/project/jellyfish/images/val/029894031.jpg: 384x640 (no detections), 6.2ms
image 5/66 /home/priyanshu/project/jellyfish/images/val/029907939.jpg: 384x640 (no detections), 6.1ms
image 6/66 /home/priyanshu/project/jellyfish/images/val/030014151.jpg: 384x640 1 arctic_comb, 6.2ms
image 7/66 /home/priyanshu/project/jellyfish/images/val/030016351.jpg: 384x640 1 common_northern_comb, 7.3ms
image 8/66 /home/priyanshu/project/jellyfish/images/val/030036855.jpg: 384x640 1 arctic_comb, 6.2ms
image 9/66 /home/priyanshu/project/jellyfish/images/val/036666913.jpg: 384x640 (no detections), 8.4ms
image 10/66 /home/priyanshu/project/jellyfish/images/val/036667053.jpg: 384x640 (

## 5. Count detections → CSV
Goes through a folder of images and writes one row per detected object,
plus a per-image, per-class count. Single clean pass (no duplicated loops).

In [18]:
IMAGE_FOLDER = "images/train"          # folder to scan
OUTPUT_CSV   = "detection_results.csv"  # where results are written
CONF, IOU    = 0.4, 0.5                 # detection thresholds

rows = []  # collect every detection here first

for img_name in sorted(os.listdir(IMAGE_FOLDER)):
    if not img_name.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    results = best_model(os.path.join(IMAGE_FOLDER, img_name), conf=CONF, iou=IOU)

    # how many of each class are in THIS image
    class_counts = defaultdict(int)
    detections = []

    for r in results:
        if r.boxes is None:
            continue
        for i in range(len(r.boxes)):
            name = best_model.names[int(r.boxes.cls[i])]
            x, y, w, h = (float(v) for v in r.boxes.xywh[i])
            class_counts[name] += 1
            detections.append({
                "image_name": img_name,
                "class": name,
                "confidence": round(float(r.boxes.conf[i]), 4),
                "x_center": round(x, 4), "y_center": round(y, 4),
                "width": round(w, 4), "height": round(h, 4),
            })

    # attach the per-image count to each detection, then keep it
    for d in detections:
        d["count_per_class_in_image"] = class_counts[d["class"]]
        rows.append(d)

# write the CSV
with open(OUTPUT_CSV, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=[
        "image_name", "class", "confidence",
        "x_center", "y_center", "width", "height",
        "count_per_class_in_image",
    ])
    writer.writeheader()
    writer.writerows(rows)

print(f"Wrote {len(rows)} detections to {OUTPUT_CSV}")


image 1/1 /home/priyanshu/project/jellyfish/images/train/028911765.jpg: 384x640 1 moon_jelly, 7.7ms
Speed: 1.2ms preprocess, 7.7ms inference, 5.1ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 /home/priyanshu/project/jellyfish/images/train/028985529.jpg: 384x640 1 common_northern_comb, 6.1ms
Speed: 1.4ms preprocess, 6.1ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 /home/priyanshu/project/jellyfish/images/train/029500279.jpg: 384x640 1 arctic_comb, 6.1ms
Speed: 1.3ms preprocess, 6.1ms inference, 6.2ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 /home/priyanshu/project/jellyfish/images/train/029673729.jpg: 384x640 1 moon_jelly, 6.0ms
Speed: 0.8ms preprocess, 6.0ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 /home/priyanshu/project/jellyfish/images/train/029674819.jpg: 384x640 1 moon_jelly, 5.9ms
Speed: 0.8ms preprocess, 5.9ms inference, 4.8ms postprocess per image at shape (1, 3, 384, 640)

ima